In [0]:
catalog = "sandbox-rey-01"
schemas = ["bronze", "silver", "gold"]
external_location = "loc_sandbox_sblakera"

In [0]:
base_url = spark.sql(f"DESCRIBE EXTERNAL LOCATION `{external_location}`").select("URL").collect()[0][0]
paths = {schema: f"{base_url}medallion/{schema}" for schema in schemas}

processed_paths = {}

for schema, path in paths.items():
    processed_paths[schema] = path

print(processed_paths)


{'bronze': 'abfss://tele-sandbox@sblakera.dfs.core.windows.net/medallion/bronze', 'silver': 'abfss://tele-sandbox@sblakera.dfs.core.windows.net/medallion/silver', 'gold': 'abfss://tele-sandbox@sblakera.dfs.core.windows.net/medallion/gold'}


In [0]:
def create_schema(catalog, path, schema):
    print(f"""Using {catalog} """)
    spark.sql(f""" USE CATALOG `{catalog}`""")
    print(f"""Creating {schema} Schema in {catalog}""")
    spark.sql(f"""CREATE SCHEMA IF NOT EXISTS `{schema}` MANAGED LOCATION '{path}'""")


In [0]:
for schema, processed_path in processed_paths.items():
    create_schema(catalog, processed_path, schema)

Using sandbox-rey-01 


---------------------------------------------------------------------------
ParseException                            Traceback (most recent call last)
File <command-7796179890298936>, line 2
      1 for schema, processed_path in processed_paths.items():
----> 2     create_schema(catalog, processed_path, schema)

File <command-7796179890298934>, line 3, in create_schema(catalog, path, schema)
      1 def create_schema(catalog, path, schema):
      2     print(f"""Using {catalog} """)
----> 3     spark.sql(f""" USE CATALOG '{catalog}'""")
      4     print(f"""Creating {schema} Schema in {catalog}""")
      5     spark.sql(f"""CREATE SCHEMA IF NOT EXISTS `{schema}` MANAGED LOCATION '{path}'""")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.cl